# Deep Learning with PyTorch · Capstone project: a CNN that reads handwritten digits

Macademia · For adults

> **Running in Colab:** Open this file at [colab.research.google.com](https://colab.research.google.com) with **File → Upload notebook**. For a GPU choose **Runtime → Change runtime type → T4 GPU**. Run the cells from top to bottom (Shift + Enter). PyTorch is preinstalled in Colab; to run on your own computer use `pip install torch torchvision`.

In [ ]:
import torch, random, numpy as np
torch.manual_seed(0); random.seed(0); np.random.seed(0)          # reproducibility
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('PyTorch', torch.__version__, '| device:', device)

## Plan
1. Data and validation split  2. Model  3. Training + checkpoint  4. Evaluation (confusion matrix)  5. Export (JSON + ONNX)  6. Try it in the browser

In [ ]:
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
tf = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))])
tr = datasets.MNIST('veri', train=True, download=True, transform=tf)
te = datasets.MNIST('veri', train=False, download=True, transform=tf)
tr_dl = DataLoader(tr, batch_size=64, shuffle=True); te_dl = DataLoader(te, batch_size=1000)
print(len(tr), len(te))

In [ ]:
from torch import nn
class KucukCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 8, 3, padding=1)
        self.conv2 = nn.Conv2d(8, 16, 3, padding=1)
        self.pool = nn.MaxPool2d(2)
        self.fc = nn.Linear(16 * 7 * 7, 10)
    def forward(self, x):
        x = self.pool(torch.relu(self.conv1(x)))                     # (N, 8, 14, 14)
        x = self.pool(torch.relu(self.conv2(x)))                     # (N, 16, 7, 7)
        return self.fc(x.flatten(1))                                 # (N, 10)

## Unit tests

In [ ]:
m = KucukCNN()
assert m(torch.randn(4, 1, 28, 28)).shape == (4, 10)
assert sum(p.numel() for p in m.parameters()) == 9098
print('tests passed')

In [ ]:
def evaluate(model):
    model.eval(); ok = 0
    with torch.no_grad():
        for xb, yb in te_dl:
            ok += (model(xb.to(device)).argmax(1).cpu() == yb).sum().item()
    return ok / len(te)

model = KucukCNN().to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
for epoch in range(4):
    model.train()
    for xb, yb in tr_dl:
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad(); nn.functional.cross_entropy(model(xb), yb).backward(); opt.step()
    print(f'epoch {epoch + 1}: test acc {evaluate(model):.4f}')

## Confusion matrix and the most confused pairs

In [ ]:
import matplotlib.pyplot as plt
cm = torch.zeros(10, 10, dtype=torch.long); wrong = []
model.eval()
with torch.no_grad():
    for xb, yb in te_dl:
        p = model(xb.to(device)).argmax(1).cpu()
        for t, q in zip(yb, p):
            cm[t, q] += 1
plt.imshow(cm, cmap='Blues'); plt.colorbar(); plt.xlabel('predicted'); plt.ylabel('true'); plt.show()
off = cm.clone(); off.fill_diagonal_(0)
top = torch.topk(off.flatten(), 3)
for v, i in zip(top.values, top.indices):
    print(f'{i.item() // 10} -> {i.item() % 10}: {v.item()}')

## Export
Download `agirliklar.json` and give it to the recogniser on the capstone page with **Load your own weights**.

In [ ]:
import json
r = lambda t: [round(v, 6) for v in t.detach().cpu().flatten().tolist()]
sd = model.state_dict()
json.dump({'mean': 0.1307, 'std': 0.3081, 'testAcc': round(evaluate(model), 4),
           'conv1': {'w': r(sd['conv1.weight']), 'b': r(sd['conv1.bias'])},
           'conv2': {'w': r(sd['conv2.weight']), 'b': r(sd['conv2.bias'])},
           'fc': {'w': r(sd['fc.weight']), 'b': r(sd['fc.bias'])}}, open('agirliklar.json', 'w'))
torch.onnx.export(model.cpu().eval(), torch.randn(1, 1, 28, 28), 'rakam.onnx', input_names=['x'], output_names=['logit'])
try:
    from google.colab import files
    files.download('agirliklar.json')
except ImportError:
    print('Outside Colab: the file is in this folder')

## Submission checklist
- [ ] R1: test accuracy ≥ 98%
- [ ] R2: < 50,000 parameters
- [ ] R3: confusion matrix + 3 pairs
- [ ] R4: repeat with the same seed
- [ ] R5: same predictions in the browser
- [ ] R6: your own 30 drawings